In [ ]:
# transcript extract

In [1]:
!pip install pydub faster-whisper

In [1]:
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from faster_whisper import WhisperModel
import pandas as pd

audio_path = "/gpfs/home/sd6701/COPE_audio/5_6_Still_Face.wav"
audio = AudioSegment.from_wav(audio_path)

chunks = detect_nonsilent(
    audio,
    min_silence_len=1000,
    silence_thresh=audio.dBFS - 12,
    seek_step=10
)

model = WhisperModel("small", device="cpu", compute_type="int8")

rows = []
bad_phrases = [
    "thanks for watching",
    "see you guys in the next video",
    "subscribe",
    "like and comment"
]

for i, (start_ms, end_ms) in enumerate(chunks):
    duration = end_ms - start_ms
    chunk = audio[start_ms:end_ms]

    if duration < 1000:
        continue

    if chunk.dBFS < -35:
        continue

    temp_path = f"/tmp/chunk_{i}.wav"
    chunk.export(temp_path, format="wav")

    segments, info = model.transcribe(
        temp_path,
        language="en",
        beam_size=5
    )
    segments = list(segments)

    for seg in segments:
        text = seg.text.strip()
        text_lower = text.lower()
        if not text:
            continue
        if any(p in text_lower for p in bad_phrases):
            continue
        if (duration / 1000) < 1.5 and len(text.split()) > 6:
            continue

        rows.append({
            "chunk_id": i,
            "global_start": round(start_ms / 1000 + seg.start, 2),
            "global_end": round(start_ms / 1000 + seg.end, 2),
            "text": text
        })

df = pd.DataFrame(rows)
df.to_csv("transcript_chunked_clean.csv", index=False)
df.head(30)

/gpfs/data/shenlab/yw5653/miniconda3/envs/iris-hpc/lib/python3.10/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


,chunk_id,global_start,global_end,text
0,21,127.03,130.55,"Okay, you can turn to the wall beside you for ..."
1,22,146.74,150.82,"Okay, now you can turn and look at Adam with a..."
2,33,272.84,275.84,"Okay, you can turn to the wall again for anoth..."
3,34,291.02,294.02,"Okay, you can interact with that as you normal..."


In [1]:
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from faster_whisper import WhisperModel
import pandas as pd
import os

input_dir = "/gpfs/home/sd6701/COPE_audio"
output_dir = "/gpfs/home/sd6701/COPE_audio/transcripts"
temp_dir = "/tmp/chunks_sd6701"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(temp_dir, exist_ok=True)

model = WhisperModel("small", device="cpu", compute_type="int8")

MIN_SILENCE_LEN = 1000
SEEK_STEP = 10
MIN_CHUNK_MS = 1000
MIN_DBFS = -35

bad_phrases = [
    "thanks for watching",
    "see you guys in the next video",
    "subscribe",
    "like and comment"
]

def sec_to_mmss(x):
    m = int(x // 60)
    s = x % 60
    return f"{m:02d}:{s:05.2f}"

def transcribe_one_audio(audio_path, output_csv_path):
    print(f"\nProcessing: {audio_path}")

    audio = AudioSegment.from_wav(audio_path)

    chunks = detect_nonsilent(
        audio,
        min_silence_len=MIN_SILENCE_LEN,
        silence_thresh=audio.dBFS - 12,
        seek_step=SEEK_STEP
    )

    print(f"Detected {len(chunks)} raw chunks")

    rows = []

    for i, (start_ms, end_ms) in enumerate(chunks):
        duration = end_ms - start_ms
        chunk = audio[start_ms:end_ms]

        if duration < MIN_CHUNK_MS:
            continue

        if chunk.dBFS < MIN_DBFS:
            continue

        temp_path = os.path.join(
            temp_dir,
            f"{os.path.splitext(os.path.basename(audio_path))[0]}_chunk_{i}.wav"
        )

        chunk.export(temp_path, format="wav")

        try:
            segments, info = model.transcribe(
                temp_path,
                language="en",
                beam_size=5
            )
            segments = list(segments)

            for seg in segments:
                text = seg.text.strip()
                text_lower = text.lower()

                if not text:
                    continue

                if any(p in text_lower for p in bad_phrases):
                    continue

                if (duration / 1000) < 1.5 and len(text.split()) > 6:
                    continue

                global_start = round(start_ms / 1000 + seg.start, 2)
                global_end = round(start_ms / 1000 + seg.end, 2)

                rows.append({
                    "chunk_id": i,
                    "global_start": global_start,
                    "global_end": global_end,
                    "start_mmss": sec_to_mmss(global_start),
                    "end_mmss": sec_to_mmss(global_end),
                    "text": text
                })

        except Exception as e:
            print(f"Error in chunk {i}: {e}")

        if os.path.exists(temp_path):
            os.remove(temp_path)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv_path, index=False)
    print(f"Saved: {output_csv_path}")
    print(f"Final rows: {len(df)}")

In [ ]:
for filename in os.listdir(input_dir):
    if not filename.lower().endswith(".wav"):
        continue

    audio_path = os.path.join(input_dir, filename)
    output_csv_path = os.path.join(
        output_dir,
        f"{os.path.splitext(filename)[0]}_transcript_chunked_clean.csv"
    )

    try:
        transcribe_one_audio(audio_path, output_csv_path)
    except Exception as e:
        print(f"Failed on {filename}: {e}")

In [3]:
filename = "100_6_Visual_Attention.wav"

audio_path = os.path.join(input_dir, filename)
output_csv_path = os.path.join(
    output_dir,
    f"{os.path.splitext(filename)[0]}_transcript_chunked_clean.csv"
)

transcribe_one_audio(audio_path, output_csv_path)


Processing: /gpfs/home/sd6701/COPE_audio/100_6_Visual_Attention.wav
Detected 13 raw chunks
Saved: /gpfs/home/sd6701/COPE_audio/transcripts/100_6_Visual_Attention_transcript_chunked_clean.csv
Final rows: 31


In [4]:
df = pd.read_csv("/gpfs/home/sd6701/COPE_audio/transcripts/100_6_Visual_Attention_transcript_chunked_clean.csv")
df

,chunk_id,global_start,global_end,start_mmss,end_mmss,text
0,0,0.00,2.00,00:00.00,00:02.00,You might be shifting the laptop so it's like.
1,2,4.80,9.76,00:04.80,00:09.76,of get perfect. You can see him as he is still...
2,3,9.28,10.44,00:09.28,00:10.44,"Oh, he's so cute."
3,3,12.94,14.78,00:12.94,00:14.78,"Yes, is it possible to scoot a bit closer?"
4,3,14.78,16.28,00:14.78,00:16.28,We just want to be able to see.
5,5,20.80,24.96,00:20.80,00:24.96,It was full so I'm gonna play this animation t...
6,10,37.14,41.58,00:37.14,00:41.58,great. So this video is less than a minute lon...
7,10,41.58,45.34,00:41.58,00:45.34,"totally okay. Some babies love it, some babies..."
8,10,45.34,47.66,00:45.34,00:47.66,"you can totally like readjust him, but you don..."
9,12,51.22,54.22,00:51.22,00:54.22,"And now, ladies and gentlemen, Cecile!"
